In [20]:
import json, random, difflib
from mlx_lm import load, generate
from apriori_core import load_json, mine, predict

MODEL, ADAPTER = "Qwen/Qwen2.5-0.5B-Instruct", "out_mlx"      # change ADAPTER if you renamed the folder
FIELDS = ["category", "labels", "shelf_life_days", "storage", "common_uses"]
INSTRUCTION = "Suggest the best item to add to this cart. Do not generate the outcome on your own"

ds2 = {d["item_id"]: d for d in load_json("ds2_descriptions.json")}
ID = {d["name"]: i for i, d in ds2.items()}
rules, _ = mine(load_json("ds1_train.json"), 0.004, 0.3)
model, tok = load(MODEL, adapter_path=ADAPTER)

def describe(i):
    d = ds2[i]; parts = [i, d["name"]]
    for f in FIELDS:
        v = d[f]; parts.append(f"{f}: {', '.join(map(str, v)) if isinstance(v, list) else v}")
    return " | ".join(parts)

def build_prompt(*names):
    ids = []
    for n in names:
        if n not in ID: raise ValueError(f"'{n}' not found. Did you mean: {difflib.get_close_matches(n, ID, 3)}")
        ids.append(ID[n])
    hits = predict(rules, ids, k=4)
    cats = list(dict.fromkeys(ds2[h[0]]["category"] for h in hits))
    cand = list(dict.fromkeys(i for c in cats for i, d in ds2.items() if d["category"] == c))[:12]
    random.Random(0).shuffle(cand)
    rule_txt = "\n".join(f"{' + '.join(sorted(ds2[i]['name'] for i in h[3]))} => {ds2[h[0]]['name']}" for h in hits) or "(no rule matched)"
    prompt = (INSTRUCTION + "\n\nCART:\n" + "\n".join(describe(i) for i in ids) + f"\n\nMINED RULES:\n{rule_txt}\n\n"
              + ("CANDIDATES:\n" + "\n".join(describe(i) for i in cand) + "\n\n" if cand else "") + "ANSWER:")
    return prompt, hits

def ask(*names, show_prompt=False):
    prompt, hits = build_prompt(*names)
    if getattr(tok, "chat_template", None):
        text = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, tokenize=False)
    else:
        text = prompt
    out = generate(model, tok, prompt=text, max_tokens=20).strip()
    if show_prompt: print(prompt, "\n")
    print("CART      :", " + ".join(names))
    print("APRIORI   :", ", ".join(f"{ds2[h[0]]['name']} ({h[1]:.2f})" for h in hits[:3]) or "-- no rule --")
    print("FINE-TUNED:", out, "\n")

# ---- one-off tests: edit freely (names must match DS2 exactly) ----
ask("Vegan Cheese Slices", "Bananas", "Sourdough") 
ask("Gluten-Free Muesli", "Almond Milk")   
ask("Sourdough", "Soy Milk")   
ask("Vegan Bolognese Sauce", "Egg Tagliatelle", "Dishwasher Tablets")                   # expect Fat-Free Butter

Fetching 7 files: 100%|██████████| 7/7 [00:00<00:00, 2018.43it/s]


CART      : Vegan Cheese Slices + Bananas + Sourdough
APRIORI   : Mayonnaise (0.47), Vegan Mayo (0.37), Eggless Mayo (0.31)
FINE-TUNED: OB_1054 | Vegan Sise S
!OB0033
_ 

CART      : Gluten-Free Muesli + Almond Milk
APRIORI   : Greek Yogurt (0.33), Coconut Yogurt (0.32)
FINE-TUNED: OB_1023 | Coconut Yogurt 

CART      : Sourdough + Soy Milk
APRIORI   : Mayonnaise (0.47)
FINE-TUNED: OB_1020 May May Saus 

CART      : Vegan Bolognese Sauce + Egg Tagliatelle + Dishwasher Tablets
APRIORI   : Parmesan (0.76)
FINE-TUNED: OB_1041 | Vegan Parmesan 



In [10]:
## Test example Vegan Burger Patties + Vegan Hot Dog Buns

text = """
Suggest the best item to add to this cart. Do not generate the outcome on your own, check the context of the cart items before you suggest.
cart = Vegan Burger Patties, Vegan Hot Dog Buns
apriori = Pork Sausages (0.60), Vegan Sausages (0.40)
"""

prompt = tok.apply_chat_template(
    [{"role": "user", "content": text}],
    add_generation_prompt=True,
    tokenize=False
)

out = generate(model, tok, prompt=prompt, max_tokens=100).strip()

print(out)

OB S S
!
OB
03arian Sarian ReOB S S03 | Maylessarian Mayoplaceronna Range May

! | Range Mayless Sarian May Rangearian Sarian
! |OB Vegan Yogausless Sausages


In [9]:

from mlx_lm import generate

text = """Suggest the best item to add to this cart. Do not generate the outcome on your own, check the context of the cart items before you suggest.

CART:
Vegan Burger Patties + Vegan Hot Dog Buns

APRIORI:
Pork Sausages (0.60)
Vegan Sausages (0.40)

ANSWER:"""

prompt = tok.apply_chat_template(
    [{"role": "user", "content": text}],
    add_generation_prompt=True,
    tokenize=False
)

out = generate(model, tok, prompt=prompt, max_tokens=32).strip()

print("FINE-TUNED:", out)


FINE-TUNED: OB
003 Veganarian Re Rangearianarian Hard Mayesanplacer Sarian

OB May |003 MayOB May0000
